In [278]:
import numpy as np
import pandas as pd

In [279]:
df = pd.read_csv("diabetes.csv")

In [280]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [281]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [282]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [283]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [284]:
X = scaler.fit_transform(X)

In [285]:
X

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]])

In [286]:
X.shape

(768, 8)

In [287]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [288]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Dropout

In [289]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=8))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [290]:
model.fit(X_train, y_train, batch_size=32, epochs=15, validation_data=(X_test, y_test))

Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6678 - loss: 0.6207 - val_accuracy: 0.7338 - val_loss: 0.6011
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7085 - loss: 0.5675 - val_accuracy: 0.7338 - val_loss: 0.5684
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7280 - loss: 0.5299 - val_accuracy: 0.7662 - val_loss: 0.5418
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7492 - loss: 0.5045 - val_accuracy: 0.7857 - val_loss: 0.5229
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4850 - val_accuracy: 0.7662 - val_loss: 0.5123
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7622 - loss: 0.4709 - val_accuracy: 0.7662 - val_loss: 0.5058
Epoch 7/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7736 - loss: 0.4589 - val_accuracy: 0.7662 - val_loss: 0.5054
Epoch 8/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4519 - val_accuracy: 0.7597 - val_loss

In [291]:
# 1. How to select appropriate optimizer
# 2. Number of Neurons in a layer
# 3. How to select Number of layers
# 4. All in one model

In [292]:
# pip install -U keras-tuner

In [293]:
import kerastuner as kt

In [294]:
 def build_model(hp):

  model = Sequential()

  model.add(Dense(32, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta'])

  model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=["accuracy"])

  return model


In [295]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5)

Reloading Tuner from ./untitled_project/tuner0.json


In [296]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [297]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [298]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [299]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [300]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7557 - loss: 0.5239 - val_accuracy: 0.7273 - val_loss: 0.5440
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7476 - loss: 0.5065 - val_accuracy: 0.7338 - val_loss: 0.5352
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7590 - loss: 0.4959 - val_accuracy: 0.7338 - val_loss: 0.5251
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7557 - loss: 0.4868 - val_accuracy: 0.7468 - val_loss: 0.5190
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4794 - val_accuracy: 0.7597 - val_loss: 0.5130
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7671 - loss: 0.4734 - val_accuracy: 0.7597 - val_loss: 0.5085
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4683 - val_accuracy: 0.7727 - val_loss: 0.5047
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.4647 - val_accuracy: 0.76

In [301]:
def build_model(hp):

  model = Sequential()

  units = hp.Int('units', min_value=8, max_value=128, step=8)

  model.add(Dense(units=units, activation='relu', input_dim=8))
  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

  return model

In [302]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir',
                        project_name='Mandar Bavdane')

Reloading Tuner from mydir/Mandar Bavdane/tuner0.json


In [303]:
tuner.search(X_train, y_train, epochs=4, validation_data=(X_test, y_test))

In [304]:
tuner.get_best_hyperparameters()[0].values

{'units': 88}

In [305]:
model = tuner.get_best_models(num_models=1)[0]

In [306]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7671 - loss: 0.4898 - val_accuracy: 0.7792 - val_loss: 0.5019
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7752 - loss: 0.4745 - val_accuracy: 0.7857 - val_loss: 0.4997
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7769 - loss: 0.4670 - val_accuracy: 0.7792 - val_loss: 0.5000
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4610 - val_accuracy: 0.7792 - val_loss: 0.5022
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4561 - val_accuracy: 0.7792 - val_loss: 0.5025
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4522 - val_accuracy: 0.7727 - val_loss: 0.5026
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4499 - val_accuracy: 0.7727 - val_loss: 0.5047
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4473 - val_accuracy: 0.7

#How to Select Number of Layers

In [307]:
def build_model(hp):
  model = Sequential()

  model.add(Dense(72, activation='relu', input_dim=8))

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

    model.add(Dense(72, activation='relu'))

  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

  return model


In [308]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=3,
                        directory='mydir',
                        project_name='num_layers')

Reloading Tuner from mydir/num_layers/tuner0.json


In [309]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [310]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 1}

In [311]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 8 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [312]:
model.fit(X_train, y_train, batch_size=32, epochs=100, initial_epoch=6, validation_data=(X_test, y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7736 - loss: 0.4641 - val_accuracy: 0.7532 - val_loss: 0.5056
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7736 - loss: 0.4511 - val_accuracy: 0.7662 - val_loss: 0.5095
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4436 - val_accuracy: 0.7662 - val_loss: 0.5116
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7915 - loss: 0.4374 - val_accuracy: 0.7468 - val_loss: 0.5205
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7932 - loss: 0.4340 - val_accuracy: 0.7662 - val_loss: 0.5195
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7948 - loss: 0.4306 - val_accuracy: 0.7597 - val_loss: 0.5185
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8046 - loss: 0.4262 - val_accuracy: 0.7662 - val_loss: 0.5237
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4207 - val_accuracy: 0.76

In [313]:
def build_model(hp):

  model = Sequential()

  counter = 0

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

    if counter == 0:

      model.add(Dense(hp.Int('units' + str(i), min_value=8, max_value=128, step=8), activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])))
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))


    else:

      model.add(Dense(hp.Int('units' + str(i), min_value=8, max_value=128, step=8), activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])))
      model.add(Dropout(hp.Choice('dropout' + str(i), values=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])))

    counter += 1

  model.add(Dense(1, activation='sigmoid'))

  model.compile(optimizer=hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop', 'adadelta', 'nadam']),
                loss='binary_crossentropy',
                metrics=['accuracy']
                )

  return model

In [314]:
tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials=5,
                        directory='mydir',
                        project_name='Full&Final')

In [315]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.7142857313156128
Total elapsed time: 00h 00m 27s


In [316]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 2,
 'units0': 104,
 'activation0': 'tanh',
 'dropout0': 0.1,
 'optimizer': 'sgd',
 'units1': 96,
 'activation1': 'sigmoid',
 'dropout1': 0.2,
 'units2': 32,
 'activation2': 'sigmoid',
 'dropout2': 0.8,
 'units3': 8,
 'activation3': 'tanh',
 'dropout3': 0.2,
 'units4': 32,
 'activation4': 'tanh',
 'dropout4': 0.4,
 'units5': 56,
 'activation5': 'tanh',
 'dropout5': 0.6,
 'units6': 112,
 'activation6': 'tanh',
 'dropout6': 0.5,
 'units7': 16,
 'activation7': 'tanh',
 'dropout7': 0.5,
 'units8': 8,
 'activation8': 'sigmoid',
 'dropout8': 0.5,
 'units9': 24,
 'activation9': 'sigmoid',
 'dropout9': 0.7}

In [317]:
model = tuner.get_best_models(num_models=1)[0]

In [277]:
model.fit(X_train, y_train, epochs=200, initial_epoch=5, validation_data=(X_test, y_test))

Epoch 6/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7296 - loss: 0.5229 - val_accuracy: 0.7468 - val_loss: 0.5282
Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7329 - loss: 0.5013 - val_accuracy: 0.7143 - val_loss: 0.5298
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.4927 - val_accuracy: 0.7013 - val_loss: 0.5260
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.4826 - val_accuracy: 0.7597 - val_loss: 0.5070
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.4734 - val_accuracy: 0.7662 - val_loss: 0.5025
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7687 - loss: 0.4678 - val_accuracy: 0.7468 - val_loss: 0.5010
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.4625 - val_accuracy: 0.7208 - val_loss: 0.5114
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4547 - val_accuracy: 0.759